    # FabricDaxLoadTest — LoadTest Main

    **This is your Load Test.** Edit cell 1 and run it. Each Run-All mints
    a fresh `RunId` so re-running is purely additive.

    ## Quickstart (4 steps, no lakehouse required)

    1. **Drop a queries `.json` onto the Resources panel** (left sidebar).
       Power BI Desktop *Performance Analyzer* exports work verbatim.
    2. **Edit cell 1**: set `TARGET_DATASET` (or leave `None` if there's
       only one semantic model in the workspace).
    3. **Run All.** Cell 3 prints a live status line; cell 4 plots
       latency / QPS / users / engine CPU.
    4. *(Optional)* set `LAKEHOUSE_NAME` in cell 1 to persist results to
       Delta tables for cross-run analysis.

    ### What's the lakehouse for?

    Charts in cell 4 read the LoadGen CSV directly from the Spark driver's
    local `/tmp/` — they need **no Spark and no lakehouse**. Setting
    `LAKEHOUSE_NAME` (cell 1) opts in to writing 5 Delta tables —
    `LoadTests`, `LoadTestRuns`, `LoadTestQueries`, `QueryExecutions`,
    `TraceEvents` — keyed so multiple runs land side-by-side and can be
    queried as a Direct Lake source for dashboards. Without it, the
    forensic artifacts (CSVs, `*.log`, `*.trace.csv`) live only on the
    driver and disappear at session end.

    > **Detailed parameter reference.** Cell 1 ships with one-line comments;
    > the full explanation of every knob (semantics, defaults, examples) is
    > in [`docs/loadgen-main.md`](../docs/loadgen-main.md).

    > **Multiple Load Tests in one workspace?** **File → Save As** /
    > **Duplicate** and rename the copy to `LoadTest - <name>`.
    > `Deploy-LoadTests.ps1` only updates the original `LoadTest - Main`,
    > so cell-1 edits on saved copies survive redeploys.

    > **Upgrades.** Change `WHEEL_URL` in cell 2 to a newer release
    > (e.g. `v0.9.0` → `v0.10.0`) and Run All. The .NET LoadGen binaries
    > ship inside the `fdlt_runtime` wheel, so there is nothing else to
    > refresh.

    ---

    Drives concurrent DAX queries against a Power BI / Fabric semantic
    model via the **XMLA endpoint** by launching `LoadGen.dll` as an
    out-of-process subprocess on the Spark driver's bundled .NET 8
    runtime. Per-run forensic artifacts (executions CSV, trace CSV,
    result.json, `*.log`) stay under `/tmp/fdlt-<RunId>/`.
    

In [ ]:
# ── 1. Configuration ──────────────────────────────────────────────────────────
# All knobs the load test reads live here. For most runs you only need to
# touch a few — see the "essential" section below. Full reference for every
# parameter (semantics, defaults, examples) is in:
#
#     docs/loadgen-main.md
#     https://github.com/dbrownems/FabricDaxLoadTest/blob/main/docs/loadgen-main.md
#
# ─────────────── Essential parameters (typical run) ───────────────────────────

# Target semantic model (None → only model in current workspace)
TARGET_DATASET   = None
TARGET_WORKSPACE = None

# Load shape
DURATION_SECONDS   = 60
CONCURRENT_USERS   = 25
USER_RAMP_TIME_SEC = 15

# Optional: persist the run to Delta tables for cross-run analysis.
# Leave None for the simplest case — charts read the local CSV.
LAKEHOUSE_NAME = None

# Scenario (queries to drive). Leave QUERIES_FILE = None to auto-pick the
# single .json attached to the notebook's *Resources* panel (e.g. a
# Power BI Performance Analyzer export). QUERIES_INLINE is the fallback.
QUERIES_FILE   = None
QUERIES_INLINE = [
    "EVALUATE ROW(\"ping\", 1)",
    "EVALUATE INFO.TABLES()",
    "EVALUATE INFO.MEASURES()",
]

# ─────────────── Advanced parameters (see docs/loadgen-main.md) ───────────────

TARGET_REPLICA               = ""        # "readonly" → scale-out read replica
LAKEHOUSE_WORKSPACE_NAME     = None      # for BYO-lakehouse in another workspace
LAKEHOUSE_SCHEMA             = None      # None → auto-detect (schema-enabled → "dbo")

LOAD_TEST_NAME               = None      # None → derived from notebook name
LOAD_TEST_DESCRIPTION        = ""        # free-text notes for this run

CONCURRENT_QUERIES_PER_USER  = 1         # in-flight queries per user (1 = serial)
PAUSE_BETWEEN_ITERATIONS_MS  = 1000      # think-time between iterations
PAUSE_BETWEEN_QUERIES_MS     = 0         # think-time between queries in an iteration

ENABLE_TRACING               = True      # capture engine events to TraceEvents
SKIP_RESULTS                 = False     # True → drain rows without parsing

USERS_FILE                   = None      # RLS / impersonation list
USERS_INLINE                 = []        # see docs/impersonation.md

LOG_FOLDER                   = None      # None → /tmp on driver; "abfss://…" or local path supported

# Runtime wheel — to upgrade, bump the version (e.g. v0.9.0 → v0.10.0) and Run-All.
WHEEL_URL = "REPLACE_ME_WITH_WHEEL_URL"

In [ ]:
# ── 2. Bootstrap: pip install the fdlt_runtime wheel and call bootstrap ──────
# As of v0.5.0 the .NET LoadGen binaries ship inside the wheel, so this
# is the entire deploy. WHEEL_URL is set in cell 1 — to upgrade, edit
# the version there and Run-All.

# The sentinel literal is constructed at runtime so the WHEEL_URL line
# in cell 1 is the *only* occurrence of the literal in the notebook source.
# That lets scripts/Deploy-LoadTests.ps1 do a blunt string-replace
# without nuking the comparison value too.
_SENTINEL = "REPLACE_ME" + "_WITH_WHEEL_URL"
if WHEEL_URL == _SENTINEL:
    raise RuntimeError(
        "Cell 1: WHEEL_URL was not patched. Either re-run the GitHub "
        "release workflow (sets FDLT_RELEASE_VERSION env var), run "
        "scripts/Deploy-LoadTests.ps1 (patches the URL to the locally "
        "uploaded wheel), or paste a release wheel URL by hand.")

import importlib, json, os, subprocess, sys, urllib.request
import notebookutils

# `*.*.*` is a "always use latest release" opt-in. Resolve it to the
# current GitHub release tag (e.g. v0.9.2) before any other URL
# handling. One GET against the public releases API; failures bubble
# up so the user knows their wildcard didn't resolve (vs. silently
# falling back to a stale pin).
if "*.*.*" in WHEEL_URL:
    _api = "https://api.github.com/repos/dbrownems/FabricDaxLoadTest/releases/latest"
    print(f"WHEEL_URL contains *.*.* — resolving latest release from {_api}")
    with urllib.request.urlopen(_api, timeout=30) as _r:
        _tag = json.loads(_r.read().decode("utf-8"))["tag_name"]
    _ver = _tag.lstrip("v")
    WHEEL_URL = WHEEL_URL.replace("v*.*.*", _tag).replace("*.*.*", _ver)
    print(f"Resolved WHEEL_URL = {WHEEL_URL}")

if WHEEL_URL.startswith("abfss://"):
    # pip requires wheel filenames to match PEP 427 (name-version-...-py3-none-any.whl);
    # strip the path component but keep the filename verbatim.
    _local = "/tmp/" + WHEEL_URL.rsplit("/", 1)[-1]
    if os.path.exists(_local):
        os.remove(_local)
    notebookutils.fs.cp(WHEEL_URL, "file://" + _local)
    _src = _local
else:
    # https://, /lakehouse/... or any other path pip already understands.
    _src = WHEEL_URL

# --no-deps: fdlt_runtime declares no Python deps today (everything it
# needs is already in the Fabric Spark image). Drop --no-deps if that
# ever changes. --force-reinstall: ensure the kernel picks up the
# just-fetched wheel even if a cached copy of the same version is
# already installed.
_pip = subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "--force-reinstall", "--no-deps", _src],
    capture_output=True, text=True, timeout=180)
if _pip.returncode != 0:
    raise RuntimeError(
        f"pip install of {_src} failed:\n{_pip.stderr or _pip.stdout}")

# Purge any cached fdlt_runtime modules so the import picks up the
# just-installed wheel (relevant on subsequent Run-All cycles when the
# kernel is reused but WHEEL_URL was bumped).
for _m in [m for m in list(sys.modules)
           if m == "fdlt_runtime" or m.startswith("fdlt_runtime.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

import fdlt_runtime
from fdlt_runtime import notebook as fdlt_nb

boot = fdlt_nb.bootstrap(
    lakehouse_name=LAKEHOUSE_NAME,
    lakehouse_workspace=LAKEHOUSE_WORKSPACE_NAME,
    lakehouse_schema=LAKEHOUSE_SCHEMA,
)

In [ ]:
# ── 3. Run the load test and persist results ─────────────────────────────────
# Thin shim: every parameter from cell 1 is passed by keyword to
# `fdlt_runtime.notebook.run()`, which loads the scenario, resolves
# the target, runs LoadGen, streams progress, and writes 4 Delta tables.
# Press the ■ Interrupt Kernel button to cancel.
outcome = fdlt_nb.run(
    boot,
    load_test_name=LOAD_TEST_NAME,
    load_test_description=LOAD_TEST_DESCRIPTION,
    target_workspace=TARGET_WORKSPACE,
    target_dataset=TARGET_DATASET,
    target_replica=TARGET_REPLICA,
    duration_seconds=DURATION_SECONDS,
    concurrent_users=CONCURRENT_USERS,
    concurrent_queries_per_user=CONCURRENT_QUERIES_PER_USER,
    pause_between_iterations_ms=PAUSE_BETWEEN_ITERATIONS_MS,
    pause_between_queries_ms=PAUSE_BETWEEN_QUERIES_MS,
    user_ramp_time_sec=USER_RAMP_TIME_SEC,
    skip_results=SKIP_RESULTS,
    enable_tracing=ENABLE_TRACING,
    queries_file=QUERIES_FILE,
    queries_inline=QUERIES_INLINE,
    users_file=USERS_FILE,
    users_inline=USERS_INLINE,
    log_folder=LOG_FOLDER,
    spark=spark,
)

In [ ]:
# ── 4. Charts ────────────────────────────────────────────────────────────────
# Latency band + QPS + active-user figure from the per-run CSV.
fdlt_nb.analyze(outcome)